# T3-05 · Chẩn đoán và thử pool Auto bổ sung
Pool ban đầu không đạt quota vì có bài 0 BibTeX và nhiều abstain. Notebook này quét BibTeX trong 5 ZIP MS2, chọn mẫu thử 50 bài/TV có metadata tốt hơn, rồi đo tỷ lệ phủ. Đây chỉ là pilot, chưa bàn giao TV5.


In [ ]:
# @title 1. Mount Drive và cập nhật code
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
REPO = Path('/content/arxiv-data-science')
BRANCH = 'feature/m3-tv3-auto-labeling'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/LeoTkiet/arxiv-data-science.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
import arxiv_pipeline
print('Code:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())


In [ ]:
# @title 2. Kiểm tra nguồn và cache 5 ZIP (tải lại nếu runtime mới)
import hashlib, json, shutil
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
BASE=Path('/content/drive/MyDrive/T3-05') # @param {type:'string'}
ROOTS=BASE/'output/roots.json'
INVENTORY=BASE/'source_inventory.json'
CACHE=Path('/content/t3-05-ms2-zips')
if not ROOTS.is_file(): raise FileNotFoundError(f'Chưa có roots.json từ notebook Prepare: {ROOTS}')
roots=json.loads(ROOTS.read_text(encoding='utf-8'))
inventory=json.loads(INVENTORY.read_text(encoding='utf-8'))['members']
assert set(roots)==set(inventory)=={f'tv{i}' for i in range(1,6)}
for m in sorted(roots):
    if not Path(roots[m]['ms1_root']).is_dir():
        raise FileNotFoundError(f'MS1 của {m} chưa hiện trong MyDrive: {roots[m]["ms1_root"]}')
auth.authenticate_user()
credentials,_=google.auth.default()
api=build('drive','v3',credentials=credentials,cache_discovery=False)
def md5_file(path):
    h=hashlib.md5()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
info={}
valid={}
for m in sorted(inventory):
    file_id=inventory[m]['ms2_results_zip_id']
    row=api.files().get(fileId=file_id,fields='id,name,size,md5Checksum',supportsAllDrives=True).execute()
    if row['name']!='MS2_RESULTS.zip': raise ValueError(f'Sai file MS2 của {m}: {row}')
    info[m]=row
    target=CACHE/m/'MS2_RESULTS.zip'
    valid[m]=target.is_file() and target.stat().st_size==int(row['size']) and (not row.get('md5Checksum') or md5_file(target)==row['md5Checksum'])
needed=sum(int(info[m]['size']) for m in info if not valid[m])
if shutil.disk_usage('/content').free<needed+2_000_000_000: raise OSError(f'Ổ tạm Colab cần thêm khoảng {needed/1e9+2:.1f} GB')
for m,row in sorted(info.items()):
    target=CACHE/m/'MS2_RESULTS.zip'
    target.parent.mkdir(parents=True,exist_ok=True)
    if not valid[m]:
        partial=target.with_suffix('.partial')
        if partial.exists():partial.unlink()
        request=api.files().get_media(fileId=row['id'],supportsAllDrives=True)
        with partial.open('wb') as f:
            dl=MediaIoBaseDownload(f,request,chunksize=32*1024*1024)
            done=False
            while not done:
                status,done=dl.next_chunk(num_retries=3)
                print(m,f'{status.progress():.0%}',end='\r')
        if partial.stat().st_size!=int(row['size']) or (row.get('md5Checksum') and md5_file(partial)!=row['md5Checksum']):
            raise ValueError(f'ZIP MS2 của {m} sai size hoặc MD5')
        partial.replace(target)
    roots[m]['ms2_root']=str(target)
ROOTS.write_text(json.dumps(roots,indent=2),encoding='utf-8')
print('Đã có 5 ZIP, không cần giải nén')


In [ ]:
# @title 3. Quét metadata và chuẩn bị pilot mở rộng (tối đa 800 bài/TV)
from arxiv_pipeline.matching.silver_candidate_scan import scan_candidates
PILOT=BASE/'output/diagnostic'
report=scan_candidates(BASE/'population_manifest.json',BASE/'output/selection.json',ROOTS,REPO/'configs/ms1_group6_ranges.json',PILOT,pilot_per_member=800)
prepared=prepare_pool(PILOT/'pilot_selection.json',ROOTS,PILOT/'prepared')
print('Pilot ready:',prepared['ready'],'/',prepared['total'])
print('Selection report:',PILOT/'candidate_scan.json')
print('Đã lấy mẫu phân tầng 800 bài/TV; đây vẫn là pilot mở rộng, chưa phải nhãn bàn giao.')
print('Manifest:',PILOT/'prepared/prepared_manifest.json')


In [ ]:
# @title 4. Đo tỷ lệ phủ heuristic trên pilot
from arxiv_pipeline.matching.silver_first_pass import run_first_pass
BASE=Path('/content/drive/MyDrive/T3-05') # @param {type:'string'}
PILOT=BASE/'output/diagnostic'
summary=run_first_pass(PILOT/'prepared/prepared_manifest.json',BASE/'population_manifest.json',PILOT/'first_pass',progress_every=50)
print('Pilot ready:',summary['prepared_ready'])
print('Bài phủ đủ tạm thời:',summary['candidate_full_coverage_publications'])
print('Phủ đủ, có references:', summary['full_coverage_with_references'])
print('Phủ đủ, không có references:', summary['full_coverage_without_references'])
print('Phủ đủ và có matched:', summary['full_coverage_with_match'])
print('Dòng uncertain:',summary['uncertain_entries'])
print('Báo cáo:',PILOT/'first_pass/first_pass_manifest.json')
print('Nguyên nhân uncertain:',summary['uncertain_reason_counts'])
print('Mẫu spot-check accepted:',summary['spot_check_sample_count'])
print('Chi tiết uncertain:',PILOT/'first_pass/uncertain_details.json')
print('Mẫu kiểm tra:',PILOT/'first_pass/spot_check_sample.json')
print('Heuristic version:',summary['policy']['heuristic_version'])
print('Spot-check target:',summary['spot_check_sample_target'])
print('Spot-check method:',summary['spot_check_sample_method'])


In [ ]:
# @title 5. Thử ngưỡng unmatched 0.40 trên cùng pilot (chưa bàn giao)
import json, hashlib
from arxiv_pipeline.matching.heuristics import HeuristicPolicy
from arxiv_pipeline.matching.silver_first_pass import run_first_pass
from arxiv_pipeline.common.utils import atomic_json
BASE=Path('/content/drive/MyDrive/T3-05')
PILOT=BASE/'output/diagnostic'
OLD=PILOT/'first_pass'
TRIAL=PILOT/'threshold_040_trial'
old=json.loads((OLD/'first_pass_manifest.json').read_text(encoding='utf-8'))
assert old['policy']['no_match_title_threshold']==0.25 and old['policy']['heuristic_version']=='v1.2.0'
trial_policy=HeuristicPolicy(heuristic_version='v1.2.0-threshold-040-trial', no_match_title_threshold=0.40, accept_clear_decisions=True)
new=run_first_pass(PILOT/'prepared/prepared_manifest.json',BASE/'population_manifest.json',TRIAL,progress_every=100,policy=trial_policy)
old_labels=json.loads((OLD/'draft_labels.json').read_text(encoding='utf-8'))
new_labels=json.loads((TRIAL/'draft_labels.json').read_text(encoding='utf-8'))
old_by_key={(r['arxiv_id'],r['citation_key']):r for r in old_labels}
assert len(old_by_key)==len(old_labels)==len(new_labels)
member_by_paper={p['arxiv_id']:p['member'] for p in new['publications']}
changed=[]
for row in new_labels:
    prior=old_by_key[(row['arxiv_id'],row['citation_key'])]
    if row['decision']!=prior['decision'] or row['groundtruth_arxiv_id']!=prior['groundtruth_arxiv_id']:
        assert prior['decision']=='uncertain' and row['decision']=='unmatched' and row['groundtruth_arxiv_id'] is None
        changed.append({'arxiv_id':row['arxiv_id'],'citation_key':row['citation_key'],'member':member_by_paper[row['arxiv_id']],'old_rule':prior['notes'],'new_rule':row['notes']})
changed.sort(key=lambda r:hashlib.sha256(f"{r['arxiv_id']}:{r['citation_key']}".encode()).hexdigest())
atomic_json(TRIAL/'new_unmatched_rows.json',changed)
summary={'baseline_full':old['candidate_full_coverage_publications'],'trial_full':new['candidate_full_coverage_publications'],'baseline_with_references':old['full_coverage_with_references'],'trial_with_references':new['full_coverage_with_references'],'baseline_with_match':old['full_coverage_with_match'],'trial_with_match':new['full_coverage_with_match'],'new_unmatched_rows':len(changed),'audit_sample_count':min(100,len(changed)),'ready_for_handoff':False}
atomic_json(TRIAL/'comparison.json',summary)
print(summary)
print('Đọc 100 key đầu của new_unmatched_rows.json và đối chiếu uncertain_details.json trước khi chốt ngưỡng.')
print('So sánh:',TRIAL/'comparison.json')
